# EUDR DeepLabV3 Training — Kaggle GPU

Single-image segmentation baseline (M1a). Trains on 2020 Sentinel-2 composites with Hansen GFC labels.

**GPU:** T4 x2 or P100 — enable via *Settings → Accelerator*

### Kaggle Datasets to attach:
- `eudr-satellite` — contains `2020_baseline/`, `2024_current/`, `hansen_labels/`, `farms_osm.csv`

### Kaggle Secrets required:
- `GEE_PROJECT_ID`

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB")

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/rajul-kk/EUDR-compliance.git"
BRANCH   = "Tessera-head"
WORK_DIR = "/kaggle/working/eudr"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
secrets = UserSecretsClient()
os.environ["GEE_PROJECT_ID"] = secrets.get_secret("GEE_PROJECT_ID")
print("Credentials loaded.")

In [ ]:
DATA = "/kaggle/input/eudr-satellite"
T1_DIR    = f"{DATA}/2020_baseline"
LABEL_DIR = f"{DATA}/hansen_labels"

import os
os.makedirs("models", exist_ok=True)

for p in [T1_DIR, LABEL_DIR]:
    print(f"[{'OK' if os.path.exists(p) else 'MISSING'}] {p}")

In [ ]:
# Train DeepLabV3 (M1a) — single-image segmentation on 2020 baseline
# DataParallel activates automatically when GPU count > 1
# batch-size 8 = 4 per T4 GPU
!python src/ML_farm_net.py \
    --raw-dir   {T1_DIR} \
    --mask-dir  {LABEL_DIR} \
    --output-model-path /kaggle/working/models/farm_deeplab.pth \
    --epochs        15 \
    --batch-size     8 \
    --learning-rate 1e-4 \
    --seed          42

In [ ]:
import torch
ckpt = torch.load("/kaggle/working/models/farm_deeplab.pth", map_location="cpu")
print("Checkpoint keys:", list(ckpt.keys()))

In [ ]:
!zip /kaggle/working/farm_deeplab.zip /kaggle/working/models/farm_deeplab.pth
print("Download farm_deeplab.zip from the Output panel.")